# 🩺 Clinical Note Intelligence
### Topic 1 — Clinical Natural Language Technology: Past → Present → Future

Cotiviti GenAI Science Internship — demonstration notebook.

This notebook turns an **unstructured clinical note** into **codeable ICD-10 / HCC
findings**, walking through three eras of clinical NLP on the same note:

1. **Past** — rule-based dictionary + negation (classic clinical NLP).
2. **Present** — LLM structured extraction with Claude.
3. **Future / LMM** — Claude *vision* reads a scanned image of the note.

All notes are **synthetic — no real PHI**.


## Setup

Run locally from the repo root (`jupyter notebook`), or on Google Colab.
On Colab, clone the repo first (replace the URL with your public repo):

```python
!git clone https://github.com/<your-user>/<your-repo>.git
%cd <your-repo>
```

For **live** Claude calls, set your key: `os.environ['ANTHROPIC_API_KEY'] = '...'`.
Without a key the notebook runs in clearly-labelled offline-simulation mode.


In [ ]:
!pip -q install anthropic pillow pyyaml python-dotenv pydantic  # no-op if already installed

In [ ]:
import os, sys
# Make the repo root importable whether run from ./ or ./notebooks
for cand in [os.getcwd(), os.path.abspath('..')]:
    if os.path.isdir(os.path.join(cand, 'src')):
        sys.path.insert(0, cand); os.chdir(cand); break

# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # <- uncomment for live mode
from src.config import have_api_key, model
print('working dir:', os.getcwd())
print('mode:', f'LIVE ({model()})' if have_api_key() else 'OFFLINE simulation')

## 1. The data

A synthetic progress note — the kind of free text where diagnoses, and the
dollars and quality gaps tied to them, are buried.


In [ ]:
note = open('data/notes/note1.txt').read()
print(note)

## 2. PAST — rule-based dictionary + negation

Curated term list, regex matching, and a shallow negation heuristic. Fast and
transparent, but brittle: it only finds terms it was told about, and it looks
only *backward* for negation cues — so it misreads *'hypertension ... suspected,
not yet confirmed'* as active.


In [ ]:
from src import rules_ner

def show(findings):
    for f in findings:
        print(f"  {f['name']:<40} {f['icd10']:<8} {f['status']:<18} conf={f['confidence']:.2f}")

show(rules_ner.extract(note))

## 3. PRESENT — LLM structured extraction (Claude)

Claude returns a structured list of conditions with ICD-10 codes, status,
the evidence span it relied on, and a calibrated confidence — normalising
abbreviations and reasoning about negation/uncertainty in context.


In [ ]:
from src import llm_extract
show(llm_extract.extract(note))

## 4. FUTURE / LMM — multimodal vision on a scanned note

A large *multimodal* model removes the classic OCR → NLP pipeline: we render
the note as a scanned image and Claude reads it directly via vision.


In [ ]:
from src import scan, vision_extract
from PIL import Image

scan_path = 'data/notes/scanned_note.png'
scan.render(note, scan_path)
display(Image.open(scan_path))
show(vision_extract.extract(scan_path, offline_text=note))

## 5. Evaluation

We score each pass against a small labeled gold set (`data/test_cases.csv`).
With a live key, the LLM's *status* accuracy exceeds the rule-based baseline —
the numbers behind the Past-vs-Present story.


In [ ]:
from src import evaluation
evaluation.evaluate()

## Architecture

![architecture](../assets/diagrams/architecture.png)

![maturity ladder](../assets/diagrams/maturity_ladder.png)


## Takeaway for Cotiviti

The same note, three eras: rule-based extraction is transparent but brittle;
the LLM normalises and reasons; the multimodal model removes the OCR stage.
For risk adjustment and payment integrity, the win is **evidence-linked,
auditable** extraction — pairing an LLM with a deterministic check against
ICD-10/HCC terminologies. See the report for the full analysis and recommendations.
